# 🚀 VIDEO-FREE · Kaggle 2 · Director (2 GPU a la vez)

Hace el video completo con las **dos GPU T4** de Kaggle trabajando a la vez: fotogramas fotorrealistas (RealVisXL), movimiento (FastWan 2.2), labios (LatentSync) y montaje con la canción. Las tomas siguen la letra.

- **Solo Kaggle (lo normal):** deja `TRABAJADORES = 2`, `PRIMER_TRABAJADOR = 1` y `MONTAR_AQUI = True`.
- **Con Colab a la vez (3 GPU):** aquí `TRABAJADORES = 3`, `PRIMER_TRABAJADOR = 2`, `MONTAR_AQUI = False`; en el cuaderno 5 de Colab `TRABAJADORES = 3`, `TRABAJADOR = 1`.
- **Puedes cortar y seguir:** cada toma se sube a tu repo apenas queda lista. Si la sesión se corta, vuelve a ejecutar: solo hace lo que falta.

Primero corre **Kaggle 1 · Canción**, para que el video use la canción completa.

**Preparar Kaggle (una sola vez):**
1. En kaggle.com, *Settings → Phone verification*: **verifica tu teléfono**. Sin eso Kaggle no da GPU ni internet.
2. *Create → New Notebook → File → Import Notebook* y sube este cuaderno (lo descargas de tu repo, carpeta `colab/kaggle/`).
3. Panel derecho → *Session options*: **Accelerator → GPU T4 x2** e **Internet → On**.
4. *Add-ons → Secrets → Add secret*: nombre `GITHUB_TOKEN` y el mismo token de GitHub que usaste en Colab. Marca la casilla junto al secreto para que este cuaderno lo use.

Luego *Run All*. Kaggle da unas **30 horas de GPU por semana** y se recargan cada semana. Kaggle no tiene Google Drive: todo se guarda en tu repo de GitHub.


In [ ]:
# 1 · Opciones (iguales a las del cuaderno 5)
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"
RAMA = "colab-ia"
PROYECTO = "carretera-sin-ti.json"
CALIDAD = "borrador"          # "borrador" (480p) o "final"
FORMATO = "horizontal"        # "horizontal" o "vertical"
MOTOR = "rapido"              # "rapido" (FastWan, 3 pasos) o "calidad" (Wan, 30 pasos)
MODO_PERSONAJE = "recorte"    # "recorte" o "entrenado"
LABIOS = True
RITMO = "auto"
SEMILLA = 11
TRABAJADORES = 2              # 2 = solo Kaggle · 3 = Colab + estas 2 GPU
PRIMER_TRABAJADOR = 1         # 1 si trabajas solo en Kaggle · 2 si Colab es el 1
MONTAR_AQUI = True            # False si Colab monta el video

import os, sys, subprocess, json, shutil
os.environ.update(VIDEOFREE_TRABAJO="/tmp/videofree/trabajo", VIDEOFREE_HERRAMIENTAS="/tmp/videofree", HF_HOME="/tmp/videofree/hf",
                  VIDEOFREE_REPO=REPO, VIDEOFREE_RAMA=RAMA)
os.makedirs("/tmp/videofree", exist_ok=True)
if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True)
if os.path.exists("/tmp/videofree/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/tmp/videofree/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/tmp/videofree/VIDEO-FREE"], check=True)
sys.path.insert(0, "/tmp/videofree/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA, usar_drive=False)
if comun._token:
    os.environ["GITHUB_TOKEN"] = comun._token  # para los dos trabajadores
GPUS = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print(len(GPUS), "GPU:", *GPUS, sep="\n  ")
OPCIONES = dict(proyecto=PROYECTO, formato=FORMATO, calidad=CALIDAD, ritmo=RITMO, semilla=SEMILLA, motor=MOTOR,
                modo_personaje=MODO_PERSONAJE, labios=LABIOS)

In [ ]:
# 2 · Instalar y descargar los modelos una sola vez para las dos GPU (8–15 min la primera vez)
!pip install -q -U "diffusers>=0.35" transformers accelerate peft ftfy sentencepiece imageio imageio-ffmpeg
import importlib, director, generar
importlib.reload(director); importlib.reload(generar)
from huggingface_hub import snapshot_download
SDXL = ["model_index.json", "scheduler/*", "tokenizer/*", "tokenizer_2/*", "*/config.json", "*.fp16.safetensors"]
snapshot_download(generar.INPAINT, allow_patterns=SDXL)
snapshot_download(generar.REALVIS_RAPIDO, allow_patterns=SDXL)
snapshot_download(generar.M5, allow_patterns=["model_index.json", "scheduler/*", "tokenizer/*", "text_encoder/*", "vae/*"]
                  + ([] if MOTOR == "rapido" else ["transformer/*"]))
if MOTOR == "rapido":
    snapshot_download(generar.FAST, allow_patterns=["transformer/*"])
if LABIOS:
    generar.Director.instalar_latentsync(vram=15)
print("Todo listo para las dos GPU")

In [ ]:
# 3 · Las dos GPU trabajan a la vez (cada una en sus tomas)
import time
procesos = []
for i in range(min(2, len(GPUS))):
    k = PRIMER_TRABAJADOR + i
    registro = open(f"/tmp/videofree/trabajador_{k}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(i), "VIDEOFREE_TRABAJO": f"/tmp/videofree/trabajo_{k}"}
    procesos.append((k, subprocess.Popen([sys.executable, "-u", "/tmp/videofree/VIDEO-FREE/colab/generar.py", "--config", json.dumps(OPCIONES),
                                          "--trabajador", str(k), "--trabajadores", str(TRABAJADORES), "--repo", REPO, "--rama", RAMA],
                                         cwd="/tmp/videofree/VIDEO-FREE/colab", env=env, stdout=registro, stderr=subprocess.STDOUT)))
print("Trabajadores en marcha:", [k for k, _ in procesos])
while any(p.poll() is None for _, p in procesos):
    time.sleep(60)
    for k, p in procesos:
        ultimas = open(f"/tmp/videofree/trabajador_{k}.log").read().strip().splitlines()[-2:]
        print(f"[{time.strftime('%H:%M')}] trabajador {k} ({'trabajando' if p.poll() is None else 'terminó'}):", *ultimas, sep="\n   ")
for k, p in procesos:
    print(f"Trabajador {k}: {'bien' if p.returncode == 0 else 'falló, mira /tmp/videofree/trabajador_%d.log' % k}")

In [ ]:
# 4 · Montaje (solo si trabajas solo en Kaggle; con Colab, monta Colab)
if MONTAR_AQUI:
    d = generar.Director(generar.Config(**OPCIONES, trabajador=PRIMER_TRABAJADOR, trabajadores=TRABAJADORES), 15)
    video = d.montar(esperar_min=30)
    d.guardar(True)
    from IPython.display import Video, display
    display(Video(str(video), embed=True, width=720))
else:
    print("Listo. En Colab, la celda 7 del cuaderno 5 monta el video con las tomas de las 3 GPU.")